# RAG Pipeline — BigQuery AI Functions

A complete Retrieval-Augmented Generation (RAG) pipeline built entirely in BigQuery SQL:

1. **Generate** a knowledge base with `AI.GENERATE_TABLE`
2. **Embed** documents with `AI.EMBED`
3. **Search** for relevant context with `VECTOR_SEARCH`
4. **Answer** questions with `AI.GENERATE`, grounded in retrieved documents

**What this demonstrates:**
- Building a full RAG system without leaving BigQuery
- Asymmetric embedding pattern (RETRIEVAL_DOCUMENT / RETRIEVAL_QUERY)
- Composing search results into grounded prompts
- Comparing RAG answers with and without retrieved context

**Functions used:** [`AI.GENERATE_TABLE`](../../functions/ai_generate_table/) | [`AI.EMBED`](../../functions/ai_embed/) | [`VECTOR_SEARCH`](../../functions/vector_search/) | [`AI.GENERATE`](../../functions/ai_generate/)

**Prerequisites:** [Setup guide](../../setup/) | [Function reference](../../RESOURCES.md)

---
## Setup

Set your project and location, authenticate, and create shared resources.

> This workflow requires a connection and a remote model for `AI.GENERATE_TABLE`. See the [Setup Reference](../../setup/) for details.

In [10]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery dataset location
DATASET_ID = 'bq_ai_functions'  # Shared dataset across all notebooks
CONNECTION_ID = 'bq_ai_functions'  # Shared connection

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [11]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install('google-cloud-bigquery', 'db-dtypes', 'bigquery-magics', 'tqdm')

In [12]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [13]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
pd.set_option('display.max_colwidth', None)

# Create the shared dataset (idempotent)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {PROJECT_ID}.{DATASET_ID} ready')

# Register %%bigquery cell magic (auto-loaded in Colab, needed elsewhere)
%load_ext bigquery_magics

Dataset statmike-mlops-349915.bq_ai_functions ready
The bigquery_magics extension is already loaded. To reload it, use:
  %reload_ext bigquery_magics


In [14]:
import subprocess as _sp, json as _json

# Create connection (idempotent)
_sp.run(['bq', 'mk', '--connection', '--location', LOCATION,
         '--connection_type', 'CLOUD_RESOURCE',
         '--project_id', PROJECT_ID, CONNECTION_ID],
        capture_output=True, text=True)

# Get service account and grant Vertex AI User role
r = _sp.run(['bq', 'show', '--connection', '--format=json',
             '--project_id', PROJECT_ID, '--location', LOCATION, CONNECTION_ID],
            capture_output=True, text=True, check=True)
sa = _json.loads(r.stdout)['cloudResource']['serviceAccountId']
_sp.run(['gcloud', 'projects', 'add-iam-policy-binding', PROJECT_ID,
         f'--member=serviceAccount:{sa}', '--role=roles/aiplatform.user', '--quiet'],
        capture_output=True, text=True)
print(f'Connection {CONNECTION_ID} ready (SA: {sa})')

Connection bq_ai_functions ready (SA: bqcx-1026793852137-hx0g@gcp-sa-bigquery-condel.iam.gserviceaccount.com)


In [15]:
# Create remote Gemini model (idempotent)
client.query(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`
  REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
  OPTIONS (endpoint = 'gemini-2.5-flash')
''').result()
print('Model gemini_flash ready')

Model gemini_flash ready


---
## Step 1 — Generate a knowledge base with AI.GENERATE_TABLE

Use `AI.GENERATE_TABLE` to create a realistic FAQ knowledge base for a fictional cloud platform. Each input row generates one FAQ entry with a question, answer, and category.

In [16]:
output_schema = """question STRING OPTIONS(description = "The FAQ question"),
       answer STRING OPTIONS(description = "Detailed answer with technical specifics"),
       category STRING OPTIONS(description = "Category: Compute, Networking, Security, Database, DevOps, or Monitoring")"""

query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_rag_knowledge` AS
SELECT question, answer, category
FROM AI.GENERATE_TABLE(
  MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash`,
  (SELECT CONCAT(
     'Write a detailed FAQ entry about "', topic, '" for a cloud computing platform. ',
     'The answer should be 2-3 sentences with specific technical details.'
   ) AS prompt
   FROM UNNEST([
     'how to create a virtual machine',
     'setting up a load balancer',
     'configuring a firewall',
     'creating a database backup',
     'monitoring application performance',
     'setting up auto-scaling',
     'managing API keys',
     'configuring DNS records',
     'setting up CI/CD pipelines',
     'managing user permissions',
     'encrypting data at rest',
     'setting up logging and alerts'
   ]) AS topic),
  STRUCT(
    """{output_schema}""" AS output_schema
  )
)
'''
client.query(query).result()

kb = client.query(
    f'SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.workflow_rag_knowledge`'
).to_dataframe()
print(f'{len(kb)} FAQ entries generated')
kb[['category', 'question']]

12 FAQ entries generated


,category,question
0,Monitoring,How can I monitor application performance in the cloud?
1,Security,How do I manage user permissions in the cloud platform?
2,Networking,How do I set up a load balancer in the cloud?
3,Security,What is data at rest encryption and how is it implemented in a cloud computing platform?
4,Database,creating a database backup
5,Compute,How do I set up auto-scaling for my applications in the cloud?
6,Networking,How do I configure DNS records for my cloud resources?
7,DevOps,How do I set up CI/CD pipelines for my cloud applications?
8,Networking,How do I configure a firewall in a cloud computing platform?
9,Compute,How do I create a virtual machine on this cloud platform?


---
## Step 2 — Embed the knowledge base with AI.EMBED

Create embeddings for each FAQ answer using `RETRIEVAL_DOCUMENT` task type. We embed the concatenation of question + answer for richer context.

In [17]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_rag_embedded` AS
SELECT
  question, answer, category,
  (AI.EMBED(
    content => CONCAT(question, ' ', answer),
    endpoint => 'text-embedding-005',
    task_type => 'RETRIEVAL_DOCUMENT'
  )).result AS embedding
FROM `{PROJECT_ID}.{DATASET_ID}.workflow_rag_knowledge`
'''
client.query(query).result()

verify = client.query(f'''
  SELECT category, question, ARRAY_LENGTH(embedding) AS dims
  FROM `{PROJECT_ID}.{DATASET_ID}.workflow_rag_embedded`
''').to_dataframe()
print(f'All {len(verify)} entries embedded ({verify.iloc[0]["dims"]} dimensions)')

All 12 entries embedded (768 dimensions)


---
## Step 3 — Retrieve and generate (RAG)

The core RAG pattern: for each user question, retrieve the most relevant FAQ entries with `VECTOR_SEARCH`, then pass them as context to `AI.GENERATE` to produce a grounded answer.

In [20]:
user_question = 'How do I protect my application from unauthorized access?'

query = f'''
WITH retrieved AS (
  SELECT
    base.question AS faq_question,
    base.answer AS faq_answer,
    base.category,
    distance
  FROM VECTOR_SEARCH(
    TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_rag_embedded`,
    'embedding',
    query_value => (AI.EMBED(
      content => '{user_question}',
      endpoint => 'text-embedding-005',
      task_type => 'RETRIEVAL_QUERY'
    )).result,
    top_k => 3,
    distance_type => 'COSINE'
  )
),
context AS (
  SELECT STRING_AGG(
    CONCAT('Q: ', faq_question, ' | A: ', faq_answer),
    ' ||| '
  ) AS docs
  FROM retrieved
)
SELECT (AI.GENERATE(
  CONCAT(
    'You are a cloud platform support assistant. Answer the user question based ONLY on the provided context. ',
    'If the context does not contain enough information, say so. ',
    'Context: ', c.docs,
    ' --- User question: {user_question}'
  )
)).result AS answer
FROM context c
'''
df = client.query(query).to_dataframe()
print(f'Question: {user_question}\n')
print(df.iloc[0]['answer'])

Question: How do I protect my application from unauthorized access?

To protect your application from unauthorized access, you can leverage several cloud security mechanisms:

*   **API Key Management:** Securely generate, store, rotate, and revoke API keys using Identity and Access Management (IAM) services or secret managers (like AWS Secrets Manager or Azure Key Vault). Implement strict access policies following the principle of least privilege to minimize security risks.
*   **User Permission Management:** Utilize the Identity and Access Management (IAM) service to define specific access controls for users, groups, and roles. Grant permissions via granular JSON-based policies that specify allowed actions and conditions, adhering to the principle of least privilege.
*   **Firewall Configuration:** Configure firewalls using security groups or Network Access Control Lists (NACLs) to define ingress and egress rules. These rules specify allowed or denied traffic based on protocol, port 

### View the retrieved context

See which FAQ entries were retrieved and used as context for the answer.

In [21]:
query = f'''
SELECT
  base.category,
  base.question,
  base.answer,
  distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_rag_embedded`,
  'embedding',
  query_value => (AI.EMBED(
    content => '{user_question}',
    endpoint => 'text-embedding-005',
    task_type => 'RETRIEVAL_QUERY'
  )).result,
  top_k => 3,
  distance_type => 'COSINE'
)
'''
client.query(query).to_dataframe()

,category,question,answer,distance
0,Security,How do I manage API keys for my cloud computing platform?,"Managing API keys involves securely generating, storing, rotating, and revoking credentials used for programmatic access to cloud services. It is crucial to utilize dedicated Identity and Access Management (IAM) services or secret managers like AWS Secrets Manager or Azure Key Vault for storage, avoiding hardcoding or committing keys to version control. Implement strict access policies following the principle of least privilege and establish regular key rotation schedules to minimize security risks.",0.418205
1,Security,How do I manage user permissions in the cloud platform?,"User permissions are primarily managed through the Identity and Access Management (IAM) service, which allows administrators to define specific access controls for users, groups, and roles. Permissions are granted via granular JSON-based policies, specifying allowed actions (e.g., s3:GetObject, ec2:StartInstances) and conditions across various cloud resources. This enables adherence to the principle of least privilege and robust security postures.",0.448017
2,Networking,How do I configure a firewall in a cloud computing platform?,"Configuring a cloud firewall typically involves creating and managing security groups or network access control lists (NACLs) that define ingress and egress rules. These rules specify allowed or denied traffic based on protocol (TCP, UDP, ICMP), port numbers, and source/destination IP address ranges (CIDR blocks) to protect cloud resources like virtual machines and databases.",0.475860


---
## Step 4 — Batch RAG: answer multiple questions

Process multiple user questions through the RAG pipeline at once. Each question retrieves its own context and gets a tailored answer.

In [22]:
query = f'''
WITH questions AS (
  SELECT question
  FROM UNNEST([
    'How do I make my app handle more traffic automatically?',
    'What is the best way to back up my database?',
    'How do I set up monitoring for my services?'
  ]) AS question
),
retrieved AS (
  SELECT
    query.question AS user_question,
    base.question AS faq_question,
    base.answer AS faq_answer,
    distance
  FROM VECTOR_SEARCH(
    TABLE `{PROJECT_ID}.{DATASET_ID}.workflow_rag_embedded`,
    'embedding',
    (SELECT question,
       (AI.EMBED(content => question, endpoint => 'text-embedding-005',
                 task_type => 'RETRIEVAL_QUERY')).result AS embedding
     FROM questions),
    top_k => 2,
    distance_type => 'COSINE'
  )
),
context_per_question AS (
  SELECT
    user_question,
    STRING_AGG(
      CONCAT('Q: ', faq_question, ' | A: ', faq_answer),
      ' ||| '
    ) AS context
  FROM retrieved
  GROUP BY user_question
)
SELECT
  user_question,
  (AI.GENERATE(
    CONCAT(
      'Answer this question concisely based on the context below. ',
      'Context: ', context,
      ' --- Question: ', user_question
    )
  )).result AS answer
FROM context_per_question
'''
df = client.query(query).to_dataframe()
for _, row in df.iterrows():
    print(f'Q: {row["user_question"]}')
    print(f'A: {row["answer"]}\n')

Q: What is the best way to back up my database?
A: Utilize managed database services for automated daily snapshots and Point-in-Time Recovery (PITR). For manual backups, use on-demand snapshots via console/API or database-specific utilities (e.g., `mysqldump`) to export data to object storage.

Q: How do I set up monitoring for my services?
A: Utilize platform-native services (e.g., AWS CloudWatch, Azure Monitor) for host-level metrics and APM tools for deeper application insights. Centralize logs with services like AWS CloudWatch Logs and define metric or log-based alerts with notification channels (e.g., AWS SNS).

Q: How do I make my app handle more traffic automatically?
A: Set up auto-scaling by defining an Auto Scaling Group (ASG) or Managed Instance Group (MIG) with instance launch templates and network configurations, typically associated with a Load Balancer. Then, configure scaling policies based on metrics like CPU utilization to automatically adjust the number of instances 

---
## Cleanup

Drop tables created by this workflow. Shared resources (dataset, models, connection) are left for other notebooks.

In [ ]:
for table in ['workflow_rag_knowledge', 'workflow_rag_embedded']:
    client.delete_table(f'{PROJECT_ID}.{DATASET_ID}.{table}', not_found_ok=True)
    print(f'Table {table} deleted')


### Remove all shared resources

When finished with **all** notebooks, uncomment and run the cell below to delete the shared dataset and all tables, models, and other resources within it.

In [ ]:
# client.delete_dataset(
#     f'{PROJECT_ID}.{DATASET_ID}',
#     delete_contents=True,
#     not_found_ok=True
# )
# print(f'Dataset {PROJECT_ID}.{DATASET_ID} deleted')